In [2]:
#Load ios feature selected dataset for modeling:
import pandas as pd
android_selected_features = pd.read_csv("../Processed_Data/Selected_FeaturesDatasets/android_SelectedFeatures.csv")

**Train/Test Split**

In [3]:
#Preparing dataset for training:
#Select relevant columns for analysis:
y = android_selected_features['stress']
X = android_selected_features.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

Final feature set columns: Index(['Unnamed: 0', 'race_alaskan native/white', 'audio_amp_mean_ep_2',
       'act_in_vehicle_ep_0', 'race_american indian/alaska native',
       'light_mean_ep_3', 'race_american indian/white', 'sse3-4',
       'audio_amp_std_ep_2', 'pam', 'race_asian', 'race_black',
       'act_still_ep_3', 'race_more than one', 'phq4-1', 'phq4-2', 'gender',
       'phq4_score', 'race_other/hispanic', 'phq4-4', 'loc_self_dorm_dur',
       'sse3-1', 'sse3-3', 'race_white'],
      dtype='str')
Final feature set shape: (7256, 24)


In [4]:
#checking for missing values:
print("Missing values in each column:")
print(X.isnull().sum())

Missing values in each column:
Unnamed: 0                              0
race_alaskan native/white               0
audio_amp_mean_ep_2                   559
act_in_vehicle_ep_0                     0
race_american indian/alaska native      0
light_mean_ep_3                       749
race_american indian/white              0
sse3-4                                  0
audio_amp_std_ep_2                    559
pam                                     0
race_asian                              0
race_black                              0
act_still_ep_3                          0
race_more than one                      0
phq4-1                                  0
phq4-2                                  0
gender                                140
phq4_score                              0
race_other/hispanic                     0
phq4-4                                  0
loc_self_dorm_dur                     179
sse3-1                                  0
sse3-3                                  0
rac

In [5]:
#Splitting data into test and train sets to prevent data leakage:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

#Column that identifies groups (participants):
group_col = 'uid'

#80/20 training testing split, with one testing group:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=android_selected_features[group_col]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = android_selected_features[group_col].iloc[train_idx]

#Checking stress label distribution to ensure the groups are stratified:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
stress
2.0    0.345873
3.0    0.253521
1.0    0.253165
4.0    0.110537
5.0    0.036905
Name: proportion, dtype: float64

Test distribution:
stress
3.0    0.295689
2.0    0.282332
1.0    0.239830
4.0    0.100182
5.0    0.081967
Name: proportion, dtype: float64


In [6]:
import numpy as np
y_pred_baseline = np.full_like(y_test, y_train.mean())

# 3. Evaluate baseline
from sklearn.metrics import mean_squared_error
baseline_mse = mean_squared_error(y_test, y_pred_baseline)

print("Baseline MSE:", baseline_mse)

Baseline MSE: 1.450716354619882


In [7]:
#Stratified group k-fold cross-validation to evaluate model performance:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

**Random Forest Regressor - Global**

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt', 
    random_state=42,
    n_jobs=-1
)

RF_fold_rmse = []
RF_fold_mae = []
RF_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    rf_model.fit(X_fold_train, y_fold_train)
    val_preds = rf_model.predict(X_fold_val)
    
    rmse = np.sqrt(mean_squared_error(y_fold_val, val_preds))
    mae = mean_absolute_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    
    RF_fold_rmse.append(rmse)
    RF_fold_mae.append(mae)
    RF_fold_r2.append(r2)
    
    print(f"Fold {fold+1} - RMSE: {rmse:.4f}, MAE: {mae:.4f}, R^2: {r2:.4f}")

print(f"\nAverage RMSE across folds: {np.mean(RF_fold_rmse):.4f}")
print(f"Average MAE across folds: {np.mean(RF_fold_mae):.4f}")
print(f"Average R^2 across folds: {np.mean(RF_fold_r2):.4f}")

Fold 1 - RMSE: 0.9416, MAE: 0.7420, R^2: 0.3581
Fold 2 - RMSE: 0.9120, MAE: 0.7465, R^2: 0.2347
Fold 3 - RMSE: 0.8151, MAE: 0.6487, R^2: 0.4462
Fold 4 - RMSE: 0.7208, MAE: 0.5725, R^2: 0.4821
Fold 5 - RMSE: 0.8434, MAE: 0.6900, R^2: 0.3817

Average RMSE across folds: 0.8466
Average MAE across folds: 0.6800
Average R^2 across folds: 0.3806


In [9]:
#Final evaluation on the test set:
rf_model.fit(X_train, y_train)
test_preds = rf_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_mae = mean_absolute_error(y_test, test_preds)                       
test_r2 = r2_score(y_test, test_preds)
print(f"\nTest Set - RMSE: {test_rmse:.4f}, MAE: {test_mae:.4f}, R^2: {test_r2:.4f}")


Test Set - RMSE: 1.0169, MAE: 0.8093, R^2: 0.2728


In [10]:
#Functions to convert regression predictions to class labels and compute accuracy:
from sklearn.metrics import accuracy_score
import numpy as np

# Define function to convert regression predictions to class labels based on binning:
def regression_to_class(y_pred):
    # Define bin edges
    bins = [1.5, 2.5, 3.5, 4.5]
    
    # Convert to class labels 1–5
    y_class = np.digitize(y_pred, bins) + 1
    
    return y_class

# Define function to compute per-class accuracy:
def per_class_accuracy(y_true, y_pred_class):
    classes = [1, 2, 3, 4, 5]
    class_acc = {}

    for c in classes:
        idx = (y_true == c)
        
        if np.sum(idx) == 0:
            class_acc[c] = None  # or "N/A"
        else:
            class_acc[c] = np.mean(y_pred_class[idx] == c)

    return class_acc

In [11]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = rf_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.37

Per-class accuracy:
Class 1: 0.210
Class 2: 0.643
Class 3: 0.435
Class 4: 0.115
Class 5: 0.000


**LightGBM Regressor - Global**

In [12]:
#lightGBM regression model cannot handle special characters in column names, so we need to clean the feature names before training:
import re

X_train = X_train.copy()

X_train.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', col)
    for col in X_train.columns]

X_test = X_test.copy()
X_test.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', col)
    for col in X_test.columns]

In [13]:
# LightGBM regression model with stratified group k-fold cross-validation:

from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.02,
    max_depth=10,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    force_row_wise=True,
    random_state=42,
    n_jobs=-1
)

LGBM_fold_rmse = []
LGBM_fold_mae = []
LGBM_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    lgbm_model.fit(X_fold_train, y_fold_train)
    
    val_preds = lgbm_model.predict(X_fold_val)
    
    rmse = np.sqrt(mean_squared_error(y_fold_val, val_preds))
    mae = mean_absolute_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    
    LGBM_fold_rmse.append(rmse)
    LGBM_fold_mae.append(mae)
    LGBM_fold_r2.append(r2)
    
    print(f"Fold {fold+1} - RMSE: {rmse:.4f}, MAE: {mae:.4f}, R^2: {r2:.4f}")

print(f"\nAverage RMSE across folds: {np.mean(LGBM_fold_rmse):.4f}")
print(f"Average MAE across folds: {np.mean(LGBM_fold_mae):.4f}")
print(f"Average R^2 across folds: {np.mean(LGBM_fold_r2):.4f}")

[LightGBM] [Info] Total Bins 1856
[LightGBM] [Info] Number of data points in the train set: 4456, number of used features: 20
[LightGBM] [Info] Start training from score 2.354129
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Fold 1 - RMSE: 0.9851, MAE: 0.7637, R^2: 0.2974
[LightGBM] [Info] Total Bins 1856
[LightGBM] [Info] Number of data points in the train set: 4460, number of used features: 20
[LightGBM] [Info] Start training from score 2.334753
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [14]:
#Final evaluation on the test set:
lgbm_model.fit(X_train, y_train)
test_preds = lgbm_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_mae = mean_absolute_error(y_test, test_preds)                       
test_r2 = r2_score(y_test, test_preds)
print(f"\nTest Set - RMSE: {test_rmse:.4f}, MAE: {test_mae:.4f}, R^2: {test_r2:.4f}")

[LightGBM] [Info] Total Bins 1856
[LightGBM] [Info] Number of data points in the train set: 5609, number of used features: 20
[LightGBM] [Info] Start training from score 2.332145
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

Test Set - RMSE: 1.1265, MAE: 0.8659, R^2: 0.1074


In [15]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = lgbm_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.39

Per-class accuracy:
Class 1: 0.382
Class 2: 0.688
Class 3: 0.329
Class 4: 0.055
Class 5: 0.000


**XGBoost Regressor - Global**

In [16]:
#XGBoost regression model with stratifed group k-fold cross-validation:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1
)

XGB_fold_rmse = []
XGB_fold_mae = []
XGB_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    xgb_model.fit(X_fold_train, y_fold_train)
    val_preds = xgb_model.predict(X_fold_val)
    
    rmse = np.sqrt(mean_squared_error(y_fold_val, val_preds))
    mae = mean_absolute_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    
    XGB_fold_rmse.append(rmse)
    XGB_fold_mae.append(mae)
    XGB_fold_r2.append(r2)
    
    print(f"Fold {fold+1} - RMSE: {rmse:.4f}, MAE: {mae:.4f}, R^2: {r2:.4f}")

print(f"\nAverage RMSE across folds: {np.mean(XGB_fold_rmse):.4f}")
print(f"Average MAE across folds: {np.mean(XGB_fold_mae):.4f}")
print(f"Average R^2 across folds: {np.mean(XGB_fold_r2):.4f}")

Fold 1 - RMSE: 0.9778, MAE: 0.7564, R^2: 0.3078
Fold 2 - RMSE: 0.9461, MAE: 0.7753, R^2: 0.1764
Fold 3 - RMSE: 0.8415, MAE: 0.6520, R^2: 0.4097
Fold 4 - RMSE: 0.7465, MAE: 0.5853, R^2: 0.4445
Fold 5 - RMSE: 0.8290, MAE: 0.6743, R^2: 0.4027

Average RMSE across folds: 0.8682
Average MAE across folds: 0.6887
Average R^2 across folds: 0.3482


In [17]:
#Final evaluation on the test set:
xgb_model.fit(X_train, y_train)
test_preds = xgb_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_mae = mean_absolute_error(y_test, test_preds)                       
test_r2 = r2_score(y_test, test_preds)
print(f"\nTest Set - RMSE: {test_rmse:.4f}, MAE: {test_mae:.4f}, R^2: {test_r2:.4f}")


Test Set - RMSE: 1.0685, MAE: 0.8354, R^2: 0.1970


In [18]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = xgb_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.38

Per-class accuracy:
Class 1: 0.286
Class 2: 0.669
Class 3: 0.355
Class 4: 0.127
Class 5: 0.000


**XGBoost Regressor - Personalized**

In [19]:
from sklearn.model_selection import train_test_split

#Training personalized models for each participant:
unique_participants = android_selected_features['uid'].unique()

P_XGB_models = {}
P_XGB_train_metrics = {}

for participant in unique_participants:
    
    participant_data = android_selected_features[
        android_selected_features['uid'] == participant
    ].sort_values('day')  # keep time order
    
    X = participant_data.drop(columns=['stress', 'uid', 'day'])
    y = participant_data['stress']
    
    y = y.replace(5, 4)
    
    # Skip participants with too few data points
    if len(participant_data) < 15:
        continue
    
    #Time based split: 60% train, 20% val, 20% test
    split1 = int(len(X) * 0.6)
    split2 = int(len(X) * 0.8)

    X_train = X.iloc[:split1]
    y_train = y.iloc[:split1]

    X_val = X.iloc[split1:split2]
    y_val = y.iloc[split1:split2]

    X_test = X.iloc[split2:]
    y_test = y.iloc[split2:]
    
    # Model
    P_XGB_model = XGBRegressor(
        n_estimators=500,
        max_depth=10,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        random_state=42,
        n_jobs=-1
    )
    
    # Train
    P_XGB_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    
    train_preds = P_XGB_model.predict(X_train)
    val_preds = P_XGB_model.predict(X_val)
    
    #Metrics:
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    
    train_mae = mean_absolute_error(y_train, train_preds)
    val_mae = mean_absolute_error(y_val, val_preds)
    
    P_XGB_train_metrics[participant] = {
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_mae': train_mae,
        'val_mae': val_mae
    }
    
    # Store model + test data
    P_XGB_models[participant] = {
        'model': P_XGB_model,
        'X_test': X_test,
        'y_test': y_test
    }

In [20]:
# Calculate and print average training and validation RMSE + MAE
avg_train_rmse = sum(m['train_rmse'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)
avg_val_rmse = sum(m['val_rmse'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)

avg_train_mae = sum(m['train_mae'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)
avg_val_mae = sum(m['val_mae'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)

print(f"Average TRAIN RMSE: {avg_train_rmse:.4f}")
print(f"Average VALIDATION RMSE: {avg_val_rmse:.4f}")

print(f"Average TRAIN MAE: {avg_train_mae:.4f}")
print(f"Average VALIDATION MAE: {avg_val_mae:.4f}")

Average TRAIN RMSE: 0.1059
Average VALIDATION RMSE: 0.7943
Average TRAIN MAE: 0.0709
Average VALIDATION MAE: 0.6451


In [21]:
# Calculate test metrics for each personalized model:
P_XGB_test_metrics = {}

for participant, data in P_XGB_models.items():
    
    P_XGB_model = data['model']
    X_test = data['X_test']
    y_test = data['y_test']
    
    preds = P_XGB_model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    P_XGB_test_metrics[participant] = {
        'test_rmse': rmse,
        'test_mae': mae,
        'test_r2': r2
    }

In [22]:
# Calculate and print average test metrics across all personalized models:
avg_test_rmse = sum(m['test_rmse'] for m in P_XGB_test_metrics.values()) / len(P_XGB_test_metrics)
avg_test_mae = sum(m['test_mae'] for m in P_XGB_test_metrics.values()) / len(P_XGB_test_metrics)
avg_test_r2 = sum(m['test_r2'] for m in P_XGB_test_metrics.values()) / len(P_XGB_test_metrics)

print(f"\nAverage TEST RMSE (FINAL): {avg_test_rmse:.4f}")
print(f"Average TEST MAE (FINAL): {avg_test_mae:.4f}")
print(f"Average TEST R^2 (FINAL): {avg_test_r2:.4f}")


Average TEST RMSE (FINAL): 0.8213
Average TEST MAE (FINAL): 0.6661
Average TEST R^2 (FINAL): -0.3363


In [23]:
#Accuracy of personalized models:
y_pred_reg = P_XGB_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.45

Per-class accuracy:
Class 1: N/A
Class 2: 0.833
Class 3: 0.000
Class 4: 0.000
Class 5: N/A


**LightGBM Regressor - Personalized**

In [24]:
#Personalized LightGBM models for each participant:

P_LGBM_models = {}
P_LGBM_train_metrics = {}

# Loop through each participant
for participant in unique_participants:
    
    participant_data = android_selected_features[
        android_selected_features['uid'] == participant
    ].sort_values('day')  # keep time ordering
    
    X = participant_data.drop(columns=['stress', 'uid', 'day'])
    y = participant_data['stress']

    y = y.replace(5, 4)
    
    # Skip small datasets
    if len(participant_data) < 15:
        continue
    
    #Time-based split (60 / 20 / 20)
    split1 = int(len(X) * 0.6)
    split2 = int(len(X) * 0.8)

    X_train = X.iloc[:split1]
    y_train = y.iloc[:split1]

    X_val = X.iloc[split1:split2]
    y_val = y.iloc[split1:split2]

    X_test = X.iloc[split2:]
    y_test = y.iloc[split2:]

    #Removing special characters from column names for LightGBM:
    X_train.columns = [
        re.sub(r'[^A-Za-z0-9_]+', '_', col)
        for col in X_train.columns]
    X_test.columns = [
        re.sub(r'[^A-Za-z0-9_]+', '_', col)
        for col in X_test.columns]
    
    # Random Forest model
    P_lgbm_model = LGBMRegressor(
            n_estimators=500,
            learning_rate=0.02,
            max_depth=10,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            force_row_wise=True,
            random_state=42,
            n_jobs=-1
        )

    #Train
    P_lgbm_model.fit(X_train, y_train)
    
    #Predictions
    train_preds = P_lgbm_model.predict(X_train)
    val_preds = P_lgbm_model.predict(X_val)
    
    #Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    
    train_mae = mean_absolute_error(y_train, train_preds)
    val_mae = mean_absolute_error(y_val, val_preds)
    
    P_LGBM_train_metrics[participant] = {
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_mae': train_mae,
        'val_mae': val_mae
    }
    
    # Store model + test data
    P_LGBM_models[participant] = {
        'model': P_lgbm_model,
        'X_test': X_test,
        'y_test': y_test
    }

[LightGBM] [Info] Total Bins 337
[LightGBM] [Info] Number of data points in the train set: 140, number of used features: 14
[LightGBM] [Info] Start training from score 2.650000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

In [25]:
# Calculate and print average training and validation RMSE + MAE
avg_train_rmse = sum(m['train_rmse'] for m in P_LGBM_train_metrics.values()) / len(P_LGBM_train_metrics)
avg_val_rmse = sum(m['val_rmse'] for m in P_LGBM_train_metrics.values()) / len(P_LGBM_train_metrics)

avg_train_mae = sum(m['train_mae'] for m in P_LGBM_train_metrics.values()) / len(P_LGBM_train_metrics)
avg_val_mae = sum(m['val_mae'] for m in P_LGBM_train_metrics.values()) / len(P_LGBM_train_metrics)

print(f"Average TRAIN RMSE: {avg_train_rmse:.4f}")
print(f"Average VALIDATION RMSE: {avg_val_rmse:.4f}")

print(f"Average TRAIN MAE: {avg_train_mae:.4f}")
print(f"Average VALIDATION MAE: {avg_val_mae:.4f}")

Average TRAIN RMSE: 0.4606
Average VALIDATION RMSE: 0.8196
Average TRAIN MAE: 0.3642
Average VALIDATION MAE: 0.6565


In [26]:
# Calculate test metrics for each personalized model:
P_LGBM_test_metrics = {}

for participant, data in P_LGBM_models.items():
    
    P_LGBM_model = data['model']
    X_test = data['X_test']
    y_test = data['y_test']
    
    preds = P_LGBM_model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    P_LGBM_test_metrics[participant] = {
        'test_rmse': rmse,
        'test_mae': mae,
        'test_r2': r2
    }

In [27]:
# Calculate and print average test metrics across all personalized models:
avg_test_rmse = sum(m['test_rmse'] for m in P_LGBM_test_metrics.values()) / len(P_LGBM_test_metrics)
avg_test_mae = sum(m['test_mae'] for m in P_LGBM_test_metrics.values()) / len(P_LGBM_test_metrics)
avg_test_r2 = sum(m['test_r2'] for m in P_LGBM_test_metrics.values()) / len(P_LGBM_test_metrics)

print(f"\nAverage TEST RMSE (FINAL): {avg_test_rmse:.4f}")
print(f"Average TEST MAE (FINAL): {avg_test_mae:.4f}")
print(f"Average TEST R^2 (FINAL): {avg_test_r2:.4f}")


Average TEST RMSE (FINAL): 0.8018
Average TEST MAE (FINAL): 0.6477
Average TEST R^2 (FINAL): -0.2392


**Random Forest Regressor - Personalized**

In [28]:
#Personalized Random Forest models for each participant:

P_RF_models = {}
P_RF_train_metrics = {}

# Loop through each participant
for participant in unique_participants:
    
    participant_data = android_selected_features[
        android_selected_features['uid'] == participant
    ].sort_values('day')  # keep time ordering
    
    X = participant_data.drop(columns=['stress', 'uid', 'day'])
    y = participant_data['stress']

    y = y.replace(5, 4)
    
    # Skip small datasets
    if len(participant_data) < 15:
        continue
    
    #Time-based split (60 / 20 / 20)
    split1 = int(len(X) * 0.6)
    split2 = int(len(X) * 0.8)

    X_train = X.iloc[:split1]
    y_train = y.iloc[:split1]

    X_val = X.iloc[split1:split2]
    y_val = y.iloc[split1:split2]

    X_test = X.iloc[split2:]
    y_test = y.iloc[split2:]
    
    # Random Forest model
    P_RF_model = RandomForestRegressor(
        n_estimators=500,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt', 
        random_state=42,
        n_jobs=-1
    )
    
    #Train
    P_RF_model.fit(X_train, y_train)
    
    #Predictions
    train_preds = P_RF_model.predict(X_train)
    val_preds = P_RF_model.predict(X_val)
    
    #Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    
    train_mae = mean_absolute_error(y_train, train_preds)
    val_mae = mean_absolute_error(y_val, val_preds)
    
    P_RF_train_metrics[participant] = {
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_mae': train_mae,
        'val_mae': val_mae
    }
    
    # Store model + test data
    P_RF_models[participant] = {
        'model': P_RF_model,
        'X_test': X_test,
        'y_test': y_test
    }

In [29]:
# Calculate and print average training and validation RMSE + MAE
avg_train_rmse = sum(m['train_rmse'] for m in P_RF_train_metrics.values()) / len(P_RF_train_metrics)
avg_val_rmse = sum(m['val_rmse'] for m in P_RF_train_metrics.values()) / len(P_XGB_train_metrics)

avg_train_mae = sum(m['train_mae'] for m in P_RF_train_metrics.values()) / len(P_RF_train_metrics)
avg_val_mae = sum(m['val_mae'] for m in P_RF_train_metrics.values()) / len(P_RF_train_metrics)

print(f"Average TRAIN RMSE: {avg_train_rmse:.4f}")
print(f"Average VALIDATION RMSE: {avg_val_rmse:.4f}")

print(f"Average TRAIN MAE: {avg_train_mae:.4f}")
print(f"Average VALIDATION MAE: {avg_val_mae:.4f}")

Average TRAIN RMSE: 0.5763
Average VALIDATION RMSE: 0.7949
Average TRAIN MAE: 0.4639
Average VALIDATION MAE: 0.6519


In [30]:
#Calculate test metrics for each personalized model:
P_RF_test_metrics = {}

for participant, data in P_RF_models.items():
    
    P_RF_model = data['model']
    X_test = data['X_test']
    y_test = data['y_test']
    
    preds = P_RF_model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    P_RF_test_metrics[participant] = {
        'test_rmse': rmse,
        'test_mae': mae,
        'test_r2': r2
    }

In [31]:
# Calculate and print average test metrics across all personalized models:
avg_test_rmse = sum(m['test_rmse'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)
avg_test_mae = sum(m['test_mae'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)
avg_test_r2 = sum(m['test_r2'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)

print(f"\nAverage TEST RMSE (FINAL): {avg_test_rmse:.4f}")
print(f"Average TEST MAE (FINAL): {avg_test_mae:.4f}")
print(f"Average TEST R^2 (FINAL): {avg_test_r2:.4f}")


Average TEST RMSE (FINAL): 0.7727
Average TEST MAE (FINAL): 0.6329
Average TEST R^2 (FINAL): -0.1466


In [32]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = P_RF_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")

Regression → Classification Accuracy: 0.55

Per-class accuracy:
Class 1: N/A
Class 2: 1.000
Class 3: 0.000
Class 4: 0.000
Class 5: N/A
